# LifeMate Datyth — vLLM trên Colab L4 / A100
Qwen3.5-4B + adapter **p1** và Gipformer đã fine-tune của release. `ENSEMBLE = True` (cell 3) nạp thêm adapter **e1** và **synth** để bỏ phiếu theo từng bước. Giữ nguyên agent E1P ở local, không tải model về laptop.

Dùng runtime mới, L4 24GB hoặc A100; Colab Pro/Pay As You Go còn compute units. Không chạy notebook Qwen3/Whisper cũ cùng runtime. Điểm61.26 là kết quả tác giả công bố, không phải kết quả được đảm bảo ở đây. Release đã học toàn TRAIN.

Tạo Colab secret `LIFEMATE_MODEL_API_KEY` ít nhất32 ký tự, bật notebook access. Giữ secret trên máy để gọi API, không gửi trong chat. Tất cả model/download dưới `/content`, mất khi Colab reset; không tự mua compute units.

In [ ]:
import sys, subprocess, os, json, time
from pathlib import Path
try:
    from google.colab import userdata
except ImportError:
    raise RuntimeError("Chỉ chạy notebook này trên Colab; không cài model/runtime local")
subprocess.run([sys.executable, "-m", "pip", "install", "vllm==0.32.0", "sherpa-onnx==2.0.1", "soundfile==0.14.0"], check=True)
# Nếu Colab báo package đã import bị thay phiên bản, Restart session rồi chạy từ cell kế tiếp.

In [ ]:
import sys, subprocess, os, json, time
from pathlib import Path
from google.colab import userdata
import torch
assert torch.cuda.is_available(), "CUDA chưa sẵn sàng; kiểm tra runtime và restart sau pip"
gpu = torch.cuda.get_device_properties(0)
assert gpu.total_memory >= 22 * 1024**3 and torch.cuda.is_bf16_supported(), "Profile này cần L4 24GB/A100; không tự hạ quantization trên T4"
assert "vllm_process" not in globals(), "Dùng runtime mới hoặc dừng các process trước khi chạy lại"
print({"gpu": gpu.name, "vram_gib": gpu.total_memory/1024**3, "torch": torch.__version__})
SOURCE = Path("/content/lifemate-datyth")
COMMIT = "6ef76e00647bf44e14bcb084b58ef0c503584f55"
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Datyth/ZaloAI-Challenge-lifemate.git", str(SOURCE)], check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=SOURCE, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True).strip() == COMMIT
os.chdir(SOURCE)
os.environ["PYTHONPATH"] = str(SOURCE / "src")
os.environ["VLLM_NO_USAGE_STATS"] = "1"

In [ ]:
# Helper chỉ được thực thi sau preflight Colab.
# ENSEMBLE = True: tải và phục vụ thêm adapter e1 + synth cho vote 3 adapter (local: LIFEMATE_ENSEMBLE trước npm run dev).
ENSEMBLE = True
(SOURCE / "scripts/setup/get_release_inference_models.py").write_text('"""Pinned inference-only download: original Qwen3.5 base + released p1/Gipformer (+ e1/synth adapters for the vote).\n\nNo training, no model execution, no redundant original ASR. LIFEMATE_ENSEMBLE_ADAPTERS=1 also fetches llm-lora-e1 and\nllm-lora-synth, the two other members of the step-level vote (configs/release.sh ENSEMBLE).\nInstall huggingface_hub in the isolated local environment before running.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\n\nos.environ.setdefault(\'HF_HUB_DISABLE_XET\', \'1\')\nfrom huggingface_hub import snapshot_download\n\nROOT = Path(__file__).resolve().parents[2]\nRELEASE = \'02080970ccd59a2e1aeec4fd66a17640da2cddf1\'\nBASE = \'851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a\'\n\n\ndef main():\n    release = ROOT/\'outputs/release/lifemate-final\'\n    folders = [\'asr-gipformer/\', \'llm-lora-p1/\']\n    if os.environ.get(\'LIFEMATE_ENSEMBLE_ADAPTERS\') == \'1\':\n        folders += [\'llm-lora-e1/\', \'llm-lora-synth/\']\n    print(\'Downloading released\', \', \'.join(f.rstrip(\'/\') for f in folders), \'(resume supported)\', flush=True)\n    snapshot_download(\'hugebenevolence/lifemate-final\', revision=RELEASE, local_dir=release,\n        allow_patterns=[\'README.md\', \'SHA256SUMS\', *(f + \'*\' for f in folders)], max_workers=2)\n    checked = []\n    for line in (release/\'SHA256SUMS\').read_text().splitlines():\n        sha, name = line.split(maxsplit=1); name = name.lstrip(\'*\').removeprefix(\'./\')\n        if not name.startswith(tuple(folders)): continue\n        path = (release/name).resolve()\n        if not path.is_relative_to(release.resolve()): raise ValueError(\'Unsafe checksum path\')\n        h = hashlib.sha256()\n        with path.open(\'rb\') as f:\n            for block in iter(lambda:f.read(1024**2),b\'\'): h.update(block)\n        if h.hexdigest()!=sha: raise ValueError(\'Release checksum mismatch: \'+name)\n        checked.append(name)\n    if not checked: raise ValueError(\'No released file checksum verified\')\n    missing = [f for f in folders if not any(n.startswith(f) for n in checked)]\n    if missing: raise ValueError(\'No checksum verified for \' + \', \'.join(missing))\n    print(\'Released checkpoint checksums verified:\', len(checked), flush=True)\n    print(\'Downloading original Qwen3.5-4B base weights\', flush=True)\n    snapshot_download(\'Qwen/Qwen3.5-4B\', revision=BASE, local_dir=ROOT/\'models/qwen3.5-4b\', max_workers=2)\n    result={\'upstream_git\':\'6ef76e00647bf44e14bcb084b58ef0c503584f55\',\'base_revision\':BASE,\n        \'release_revision\':RELEASE,\'release_files_checked\':checked,\'runtime_verified\':False,\n        \'note\':\'Downloads are not a reproduced leaderboard result.\'}\n    (ROOT/\'outputs/local-model-download.json\').write_text(json.dumps(result,indent=2)+\'\\n\')\n    print(\'Download complete; runtime still requires a GPU smoke test.\',flush=True)\n\n\nif __name__==\'__main__\': main()\n', encoding="utf-8")
(SOURCE / "scripts/colab_gateway.py").write_text('"""Private gateway: vLLM chat on loopback + released Gipformer on Colab CPU."""\nimport base64\nimport hmac\nimport io\nimport os\nfrom pathlib import Path\nimport tempfile\nimport threading\nimport wave\n\nfrom fastapi import FastAPI, HTTPException, Request\nimport requests\n\nRELEASE = \'02080970ccd59a2e1aeec4fd66a17640da2cddf1\'\n# Served adapters a client may ask for: p1 alone, or the three members of the step-level vote (lifemate.ensemble).\nMODELS = {\'lifemate\', \'lifemate_e1\', \'lifemate_synth\'}\nCHAT_SLOTS = 3   # one in-flight request per vote member; vLLM batches them (--max-num-seqs >= 3)\n\n\ndef create_app(asr, key, context_limit=16384):\n    if len(key) < 32: raise ValueError(\'Secret must have at least 32 characters\')\n    app = FastAPI(); lock = threading.Lock(); chat_slots = threading.BoundedSemaphore(CHAT_SLOTS)\n\n    @app.middleware(\'http\')\n    async def auth(request: Request, call_next):\n        from starlette.responses import JSONResponse\n        if not hmac.compare_digest(request.headers.get(\'authorization\', \'\'), \'Bearer \'+key):\n            return JSONResponse({\'detail\':\'Unauthorized\'}, status_code=401)\n        if int(request.headers.get(\'content-length\', \'0\')) > 12*1024**2:\n            return JSONResponse({\'detail\':\'Body too large\'}, status_code=413)\n        return await call_next(request)\n\n    @app.get(\'/v1/models\')\n    def models():\n        response=requests.get(\'http://127.0.0.1:8001/v1/models\',timeout=5); response.raise_for_status()\n        return {\'data\':response.json()[\'data\'], \'release_revision\':RELEASE,\n            \'asr\':\'released-gipformer\', \'context_limit\':context_limit, \'runtime\':\'vllm-0.32.0\',\n            \'profile\':\'upstream-p1-bf16\', \'trained_on_all_train\':True}\n\n    @app.post(\'/v1/chat/completions\')\n    def chat(body: dict):\n        if body.get(\'model\') not in MODELS or body.get(\'stream\', False): raise HTTPException(400,\'Use a served LifeMate adapter, non-streaming\')\n        body={k:v for k,v in body.items() if k in {\'model\',\'messages\',\'tools\',\'tool_choice\',\'temperature\',\'seed\',\'max_tokens\',\'chat_template_kwargs\'}}\n        body.update(temperature=0,seed=42,max_tokens=512,chat_template_kwargs={\'enable_thinking\':False})\n        if not chat_slots.acquire(blocking=False): raise HTTPException(429,\'Model busy\')\n        try:\n            r=requests.post(\'http://127.0.0.1:8001/v1/chat/completions\',json=body,timeout=120)\n            if r.status_code!=200: raise HTTPException(r.status_code,\'vLLM rejected request; inspect Colab logs\')\n            return r.json()\n        finally: chat_slots.release()\n\n    @app.post(\'/v1/audio/transcriptions\')\n    def transcribe(body: dict):\n        try:\n            encoded=body[\'audio_base64\']\n            if not isinstance(encoded,str) or len(encoded)>11_000_000: raise ValueError()\n            data=base64.b64decode(encoded,validate=True)\n            with wave.open(io.BytesIO(data),\'rb\') as w:\n                if w.getframerate()!=16000 or not 0<w.getnframes()/16000<=30: raise ValueError()\n        except (KeyError,ValueError,wave.Error,EOFError): raise HTTPException(400,\'Require PCM WAV, 16kHz, <=30 seconds\')\n        if not lock.acquire(blocking=False): raise HTTPException(429,\'Model busy\')\n        try:\n            with tempfile.TemporaryDirectory() as d:\n                path=Path(d)/\'audio.wav\';path.write_bytes(data)\n                return {\'text\':asr.transcribe(path),\'release_revision\':RELEASE}\n        finally: lock.release()\n    return app\n\n\nif __name__==\'__main__\':\n    from lifemate.asr import GipformerASR\n    import uvicorn\n    asr=GipformerASR(Path(\'outputs/release/lifemate-final/asr-gipformer\'),device=\'cpu\')\n    uvicorn.run(create_app(asr,os.environ[\'LIFEMATE_MODEL_API_KEY\']),host=\'127.0.0.1\',port=8000,access_log=False)\n', encoding="utf-8")
subprocess.run([sys.executable, "scripts/setup/get_release_inference_models.py"], check=True,
    env={**os.environ, "LIFEMATE_ENSEMBLE_ADAPTERS": "1" if ENSEMBLE else "0"})
print("Pinned base + p1" + (" + e1 + synth" if ENSEMBLE else "") + " + Gipformer downloaded; release SHA256 verified.")


In [ ]:
import requests
release = SOURCE / "outputs/release/lifemate-final"
vllm_log = open("/content/lifemate-vllm.log", "w")
adapters = {"lifemate": "llm-lora-p1"}
if ENSEMBLE: adapters.update(lifemate_e1="llm-lora-e1", lifemate_synth="llm-lora-synth")
command = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", str(SOURCE / "models/qwen3.5-4b"), "--served-model-name", "qwen35_base",
    "--host", "127.0.0.1", "--port", "8001", "--dtype", "bfloat16",
    "--language-model-only", "--max-model-len", "16384", "--seed", "42",
    "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder", "--reasoning-parser", "qwen3",
    "--enable-lora", "--max-lora-rank", "32", "--max-loras", str(len(adapters)),
    "--lora-modules", *[f"{name}={release / folder}" for name, folder in adapters.items()],
    "--gpu-memory-utilization", "0.85", "--max-num-seqs", "4"]
vllm_process = subprocess.Popen(command, stdout=vllm_log, stderr=subprocess.STDOUT)
for attempt in range(180):
    if vllm_process.poll() is not None:
        print(Path("/content/lifemate-vllm.log").read_text()[-12000:]); raise RuntimeError("vLLM startup failed")
    try:
        response = requests.get("http://127.0.0.1:8001/v1/models", timeout=2)
        if response.ok and set(adapters) <= {x["id"] for x in response.json()["data"]}: break
    except requests.RequestException: pass
    time.sleep(5)
else: raise TimeoutError("vLLM not ready; inspect /content/lifemate-vllm.log")
print("vLLM ready with", sorted(adapters), response.json())

In [ ]:
key = userdata.get("LIFEMATE_MODEL_API_KEY")
assert isinstance(key, str) and len(key) >= 32
child_env = {**os.environ, "LIFEMATE_MODEL_API_KEY": key}
gateway_log = open("/content/lifemate-gateway.log", "w")
gateway_process = subprocess.Popen([sys.executable, "scripts/colab_gateway.py"], env=child_env,
    stdout=gateway_log, stderr=subprocess.STDOUT)
for attempt in range(60):
    if gateway_process.poll() is not None:
        print(Path("/content/lifemate-gateway.log").read_text()[-6000:]); raise RuntimeError("Gateway startup failed")
    try:
        response=requests.get("http://127.0.0.1:8000/v1/models", headers={"Authorization":"Bearer "+key}, timeout=3)
        if response.ok: break
    except requests.RequestException: pass
    time.sleep(2)
else: raise TimeoutError("Gateway not ready")
print(response.json())

In [ ]:
# HTTPS tunnel only. Safe to rerun without reloading Qwen/Whisper.
# Pinned checksum: https://github.com/cloudflare/cloudflared/releases/tag/2026.10.7
import hashlib
import re
import subprocess
import time
from pathlib import Path
import requests

CLOUDFLARED_VERSION = "2026.10.7"
CLOUDFLARED_SHA256 = "1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872"
CLOUDFLARED_URL = f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64"

def download_cloudflared(destination, get=requests.get, sleep=time.sleep, expected_sha256=CLOUDFLARED_SHA256):
    destination = Path(destination)
    if destination.is_file() and hashlib.sha256(destination.read_bytes()).hexdigest() == expected_sha256:
        destination.chmod(0o700)
        return destination
    for attempt in range(4):
        try:
            response = get(CLOUDFLARED_URL, timeout=(15, 120))
        except (requests.ConnectionError, requests.Timeout):
            if attempt == 3:
                raise RuntimeError("Không tải được cloudflared sau 4 lần. Giữ phiên model, chờ rồi chạy lại cell tunnel.") from None
        else:
            if response.status_code == 200:
                if hashlib.sha256(response.content).hexdigest() != expected_sha256:
                    raise RuntimeError("SHA256 cloudflared không khớp; không chạy file tải về.")
                destination.write_bytes(response.content)
                destination.chmod(0o700)
                return destination
            if response.status_code not in (429, 500, 502, 503, 504):
                response.raise_for_status()
                raise RuntimeError(f"Unexpected download response: HTTP {response.status_code}")
            if attempt == 3:
                raise RuntimeError(f"GitHub download HTTP {response.status_code} sau 4 lần; thử lại cell tunnel sau.") from None
        print(f"Download tạm thời lỗi; thử lại lần {attempt + 2}/4…")
        sleep(2 ** attempt)

active_tunnel = globals().get("tunnel_process")
tunnel_log = Path("/content/lifemate-tunnel.log")
if active_tunnel is None or active_tunnel.poll() is not None:
    cloudflared = download_cloudflared("/content/cloudflared")
    previous_stream = globals().get("tunnel_stream")
    if previous_stream is not None:
        previous_stream.close()
    tunnel_stream = tunnel_log.open("w")
    tunnel_process = subprocess.Popen(
        [str(cloudflared), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.DEVNULL, stderr=tunnel_stream,
    )
url = None
for _ in range(90):
    text = tunnel_log.read_text(errors="replace") if tunnel_log.exists() else ""
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if found:
        url = found.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError("Tunnel failed; inspect /content/lifemate-tunnel.log. Model API vẫn giữ nguyên.")
    time.sleep(1)
if not url:
    raise RuntimeError("Chưa nhận được URL; kiểm tra log rồi chạy lại cell này, không restart runtime.")
print("Dùng URL này cho LIFEMATE_MODEL_API_URL ở local:", url)
print("Dùng cùng secret đã đặt trong Colab. Không gửi khóa vào chat.")

## Gọi từ local
Dùng `.venv-local` và `python -m lifemate.hosted_runner` trong repo clone theo `docs/reviews/colab_vllm_codex_2026-11-10.md`.

Không dùng nút engine Qwen3 cũ trên dashboard với endpoint này: inventory khác, chưa nối dashboard sang runner mới. Trước tiên chạy smoke1 mẫu rồi20 mẫu; xem trace/metrics/validation. Không coi TRAIN của final p1 là held-out. Không train lại ở bước này.

**Vote 3 adapter** (khi `ENSEMBLE = True`): trước khi chạy dashboard ở local, đặt biến môi trường rồi mới `npm run dev`:

```powershell
$env:LIFEMATE_ENSEMBLE = "lifemate:E1P,lifemate_e1:E1,lifemate_synth:E1"
npm run dev
```

Không đặt biến này thì dashboard chạy p1 một mình như cũ, kể cả khi Colab phục vụ đủ 3 adapter.

In [ ]:
STOP_SERVERS = False  # Đổi True và chạy cell này khi muốn dừng.
if STOP_SERVERS:
    for process_name in ["tunnel_process", "gateway_process", "vllm_process"]:
        process = globals().get(process_name)
        if process is not None and process.poll() is None:
            process.terminate()
    print("Đã yêu cầu dừng tunnel/gateway/vLLM. Disconnect and delete runtime khi dùng xong.")
